In [4]:
import pandas as pd
import torch
import os
import numpy as np
from PIL import Image
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from nltk.tokenize import word_tokenize
from collections import Counter
import nltk
nltk.download("punkt_tab")
torch.manual_seed(88)

import json
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/sukhmanighatora/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
#Makes a dataset for the image test pairs
class CombinedDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, i):
        img_path = self.pairs["image_path"].iloc[i]
        label = self.pairs["label"].iloc[i]
        descr = self.pairs["description"].iloc[i]

        img = Image.open(img_path)
        img = img.convert("RGB")
        img = img.resize((64, 64)) 
        arr = np.array(img) / 255.0
        img_tensor = torch.tensor(arr).permute(2, 0, 1).float()

        ids_tensor = encode_text(descr)
        label_tensor = torch.tensor(label, dtype=torch.float32)

        return img_tensor, ids_tensor, label_tensor

In [3]:
#splits text into words stores as lowercase tokens 
#<unk> (1) are for words not in vocab
#keeps first 128 ids 
#less than 128 ids then rest are padded 
#turns list into pyTorch tensor
#long makes it int64
def encode_text(s):
    tokens = word_tokenize(s.lower())
    ids = []
    for word in tokens:
        ids.append(vocab.get(word,1))
    ids = ids[:128] 
    while len(ids) < 128:
        ids.append(0)
    return torch.tensor(ids, dtype=torch.long)

In [ ]:
# Load vocabulary used to train the baseline model
with open("vocab.json", "r") as f:
    vocab = json.load(f)

print("Vocabulary size:", len(vocab))

Vocabulary size: 10002


In [5]:
#encoder handles each tensor 
class encode_image(nn.Module):
    #the existing layers 
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2)
        self.flatten = nn.Flatten()
        self.linear = nn.Linear(64 * 8 * 8, 128)
    #the order the layers work 
    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.pool(self.relu(self.conv3(x)))
        x = self.flatten(x)
        x = self.linear(x)
        return x

In [6]:
#where the ids are turned into 128 number vector 
class TextEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(len(vocab), 128)
        self.fc = nn.Linear(128, 128)
        
    def forward(self, x):
        #turns ids into 128 number vector
        x = self.embedding(x)
        #the average across the tokens 
        #128 vector summarising the whole description
        x = torch.mean(x, dim=1)
        #128 vectors in 128 out and the weights are learned 
        x = self.fc(x)
        return x

In [7]:
#where image features and text features become one vector
#and decides if they are match or not match
class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(256, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 1)

    def forward(self, img, txt):
        #the vectors are glued end to end to make batch, 256
        x = torch.cat([img, txt], dim=1)
        #256 batch vector becomes 128 batch vector
        x = self.fc1(x)
        #non linearity stuff
        x = self.relu(x)
        #theres one number per sample
        x = self.fc2(x)
        return x

In [ ]:
# Creates the baseline model
img_encoder = encode_image()
txt_encoder = TextEncoder()
classifier = Classifier()

# Loads the trained model weights
ckpt = torch.load(
    "baseline_checkpoints.pt",
    map_location="cpu",
    weights_only=True
)

img_encoder.load_state_dict(ckpt["img_encoder"])
txt_encoder.load_state_dict(ckpt["txt_encoder"])
classifier.load_state_dict(ckpt["classifier"])

# Puts the model in evaluation mode
img_encoder.eval()
txt_encoder.eval()
classifier.eval()

print("Baseline model loaded successfully")

Baseline model loaded successfully


In [ ]:
# Reads the fixed test dataset
test_pairs = pd.read_csv("abo_test_pairs_40000.csv")

print("Number of test pairs:", len(test_pairs))
print(test_pairs.head())
print("Columns:", test_pairs.columns.tolist())

Number of test pairs: 40000
  image_item_id     image_id text_item_id  \
0    0615916821  616nBpxLZdL   0615916821   
1    0615916821  616nBpxLZdL   B000050415   
2    B000050415  41C4BSFPSTL   B000050415   
3    B000050415  41C4BSFPSTL   B00005041T   
4    B00005041T  41HF99W1K2L   B00005041T   

                                                text  label  
0              Full Gospel Poetry: 55 Poetic Sermons      1  
1  Lidco L200 Electric Patio Party Lights, 7 Ligh...      0  
2  Lidco L200 Electric Patio Party Lights, 7 Ligh...      1  
3  Lidco L2030 Electric Ultra Patio Lights, Firef...      0  
4  Lidco L2030 Electric Ultra Patio Lights, Firef...      1  
Columns: ['image_item_id', 'image_id', 'text_item_id', 'text', 'label']


In [ ]:
# Checks the columns in the cleaned products dataset
products = pd.read_csv("processed/cleaned_products.csv")

print(products.columns.tolist())
print(products.head())

['item_id', 'description', 'main_image_id', 'image_path']
      item_id                                        description  \
0  B000HCR89W  Classic Accessories 73922 Veranda Grill Cover,...   
1  B000SMNJ6M  18k Yellow Gold Plated Sterling Silver Genuine...   
2  B000W48HL8  Classic Accessories 70972 Veranda Patio Square...   
3  B000W4AEE6  Classic Accessories 71982 Veranda Square Ottom...   
4  B007XAKFUK  Classic Accessories 55-121-011501-00 Veranda R...   

  main_image_id                                     image_path  
0   71NWZPIdX-L  abo-images-small/images/small/19/19c1e843.jpg  
1   81Khlw4DF7L  abo-images-small/images/small/2b/2b60a1c7.jpg  
2   71j9RmPJX3L  abo-images-small/images/small/01/0190eb0b.jpg  
3   71j9RmPJX3L  abo-images-small/images/small/01/0190eb0b.jpg  
4   81sywArcBQL  abo-images-small/images/small/8b/8bf047dd.jpg  


In [ ]:
# Keeps only one image path for each image ID
image_lookup = products[
    ["main_image_id", "image_path"]
].drop_duplicates(subset="main_image_id")

# Matches the test images with their file paths
test_data = test_pairs.merge(
    image_lookup,
    left_on="image_id",
    right_on="main_image_id",
    how="left",
    validate="many_to_one"
)

# Checks the results
print("Total test pairs:", len(test_data))
print("Missing image paths:", test_data["image_path"].isna().sum())

Total test pairs: 40000
Missing image paths: 4898


In [ ]:
# Reads the original image metadata
images_metadata = pd.read_csv(
    "abo-images-small/images/metadata/images.csv.gz"
)

print(images_metadata.columns.tolist())
print(images_metadata.head())

['image_id', 'height', 'width', 'path']
      image_id  height  width             path
0  010-mllS7JL     106    106  14/14fe8812.jpg
1  01dkn0Gyx0L     122    122  da/daab0cad.jpg
2  01sUPg0387L     111    111  d2/d2daaae9.jpg
3  1168jc-5r1L     186    186  3a/3a4e88e6.jpg
4  11RUV5Fs65L      30    500  d9/d91ab9cf.jpg


In [ ]:
# Creates the full image paths
images_metadata["full_path"] = (
    "abo-images-small/images/small/"
    + images_metadata["path"]
)

# Keeps only one path for each image ID
metadata_lookup = images_metadata[
    ["image_id", "full_path"]
].drop_duplicates(subset="image_id")

# Matches the test images with the metadata
test_data = test_data.merge(
    metadata_lookup,
    on="image_id",
    how="left",
    validate="many_to_one"
)

# Fill in missing image paths
test_data["image_path"] = test_data["image_path"].fillna(
    test_data["full_path"]
)

# Checks the results
print("Total test pairs:", len(test_data))
print("Missing image paths:", test_data["image_path"].isna().sum())

Total test pairs: 40000
Missing image paths: 0


In [ ]:
# Prepares the test data for evaluation
test_data = test_data.rename(columns={"text": "description"})

# Checks the test data
print(test_data[["image_path", "description", "label"]].head())
print("Total test pairs:", len(test_data))

                                      image_path  \
0  abo-images-small/images/small/20/208bd166.jpg   
1  abo-images-small/images/small/20/208bd166.jpg   
2  abo-images-small/images/small/1c/1c4e48da.jpg   
3  abo-images-small/images/small/1c/1c4e48da.jpg   
4  abo-images-small/images/small/37/37f69f49.jpg   

                                         description  label  
0              Full Gospel Poetry: 55 Poetic Sermons      1  
1  Lidco L200 Electric Patio Party Lights, 7 Ligh...      0  
2  Lidco L200 Electric Patio Party Lights, 7 Ligh...      1  
3  Lidco L2030 Electric Ultra Patio Lights, Firef...      0  
4  Lidco L2030 Electric Ultra Patio Lights, Firef...      1  
Total test pairs: 40000


In [ ]:
# Prepares the test data for evaluation
test_dataset = CombinedDataset(test_data)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

print("Test pairs:", len(test_dataset))
print("Test batches:", len(test_loader))

Test pairs: 40000
Test batches: 625


In [ ]:
# Evaluates the baseline model on all test pairs
all_predictions = []
all_labels = []

img_encoder.eval()
txt_encoder.eval()
classifier.eval()

with torch.no_grad():
    for imgs, ids, labels in test_loader:
        img_features = img_encoder(imgs)
        txt_features = txt_encoder(ids)

        logits = classifier(img_features, txt_features).squeeze(1)
        predictions = (logits > 0).int()

        all_predictions.extend(predictions.tolist())
        all_labels.extend(labels.int().tolist())

# Calculate the first-time test accuracy
correct = sum(
    prediction == label
    for prediction, label in zip(all_predictions, all_labels)
)

accuracy = correct / len(all_labels)

print("Total test pairs evaluated:", len(all_labels))
print("First-time test accuracy:", accuracy)
print("First-time test accuracy (%):", round(accuracy * 100, 2))

Total test pairs evaluated: 40000
First-time test accuracy: 0.522125
First-time test accuracy (%): 52.21


In [ ]:
# Work out the baseline evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("Accuracy:", round(accuracy_score(all_labels, all_predictions), 4))
print("Precision:", round(precision_score(all_labels, all_predictions), 4))
print("Recall:", round(recall_score(all_labels, all_predictions), 4))
print("F1-score:", round(f1_score(all_labels, all_predictions), 4))
print("Confusion matrix:")
print(confusion_matrix(all_labels, all_predictions))

Accuracy: 0.5221
Precision: 0.5303
Recall: 0.3875
F1-score: 0.4478
Confusion matrix:
[[13135  6865]
 [12250  7750]]
